In [1]:
import os, sys, re, gzip, io, json
from datetime import datetime
import pandas as pd

In [2]:
# ---------- config from environment (00_config.sh sets RUN/REPO_ROOT/OUT_DIR/META_DIR) ----------
REPO_ROOT = os.environ.get("REPO_ROOT", os.getcwd())
RUN       = os.environ.get("RUN", "genotype_run1")
OUT_DIR   = os.environ.get("OUT_DIR", os.path.join(REPO_ROOT, "output", RUN))
META_DIR  = os.environ.get("META_DIR", os.path.join(REPO_ROOT, "metadata"))

In [3]:
SAMPLE_SHEET = os.path.join(REPO_ROOT, "input_data", "sample_sheet")
EXP_INDIR = os.path.join(REPO_ROOT, "input_data", "expression_microarray")
EXP_OUT   = os.path.join(OUT_DIR, "expr", "explore")
os.makedirs(EXP_OUT, exist_ok=True)

In [18]:
# creates a dictionary to keep the required files
FILES = {
    "tga_txt":       os.path.join(EXP_INDIR, "TGA-cohort.txt"),
    "tga_csv":       os.path.join(EXP_INDIR, "TGA-cohort-BackUp.csv"),
    "transpose_csv": os.path.join(EXP_INDIR, "transpose_numbers.csv"),
    "snp_csv": os.path.join(SAMPLE_SHEET, "All_samples_Examine_SNPs_GWAS studies_GJ-P01_infiniumSampleSheet.csv")
}

In [5]:
import os

tga_txt = FILES["tga_txt"]
assert os.path.exists(tga_txt), f"Missing file: {tga_txt}"

print("Path:", tga_txt)
print("Size (MB):", round(os.path.getsize(tga_txt)/1e6, 3))

Path: /home/sima/git_projects/illumina-genotyping-pipeline/input_data/expression_microarray/TGA-cohort.txt
Size (MB): 39.302


In [6]:
# Peek first ~10 non-empty lines raw (helps see delimiter/headers)
with open(tga_txt, "r", encoding="utf-8", errors="replace") as f:
    for i in range(10):
        line = f.readline()
        if not line: break
        print(repr(line.rstrip("\n")))

'Original\tSummarized\tDifference\tFilename\tChip Type\tScan Date\tUASG\tFinal Diagnosis\tDiagnosis Subtype\tSex\tAge At Onset\tHypertension\tDiabetes\tHypercholesterolemia\tInsulin\tOral diabetes meds\tInsulin\tGLUCOSE\tGLUCOSE2\tGLUCOSE FASTING\tHEMOGLOBIN A1C\tWHITE BLOOD CELL COUNT\tHEMOGLOBIN\tHEMATOCRIT\tPLATELET COUNT\tBLOOD UREA\tCREATININE BLOOD\tESTIMATED GFR\tGLUCOSE\tINR\taPTT\tNEUTROPHIL PERCENT\tLYMPHOCYTE PERCENT\tMONOCYTE PERCENT\tWHITE BLOOD CELL COUNT2\tHEMOGLOBIN2\tHEMATOCRIT2\tPLATELET COUNT2\tBLOOD UREA2\tCREATININE BLOOD2\tESTIMATED GFR2\tGLUCOSE2\tINR2\taPTT2\tNEUTROPHIL PERCENT2\tLYMPHOCYTE PERCENT2\tMONOCYTE PERCENT2\tCHOLESTEROL\tHDL\tLDL\tTRIGLYCERIDE\tAnticoagulation\tFinal Diagnosis\tIschemic stroke/ TIA cause\tPmHx-Stroke\tCongestive heart failure\tChronic renal insufficiency\tCancer\tUnusual Disese\ttPA\tTNK\tAny Thrombolytic (tPA or TNK)\tEVT\tNIHSS on admission (range 0-43)\tNIHSS at time of blood draw (range 0-43)\tNIHSS at 24h (range 0-43)\tCT Prior I

In [7]:
import subprocess
subprocess.run(["code", "--reuse-window", tga_txt])

CompletedProcess(args=['code', '--reuse-window', '/home/sima/git_projects/illumina-genotyping-pipeline/input_data/expression_microarray/TGA-cohort.txt'], returncode=0)

In [9]:
# Load with pandas
# Try auto-delimiter first; if it misguesses, we’ll force sep="\t" next cell
df_txt = pd.read_csv(tga_txt, sep="\t", engine="c", low_memory=False)
display(df_txt.head(5))
print("shape:", df_txt.shape)

,Original,Summarized,Difference,Filename,Chip Type,Scan Date,UASG,Final Diagnosis,Diagnosis Subtype,Sex,...,TSUnmapped00000810.hg.1,TSUnmapped00000812.hg.1,TSUnmapped00000815.hg.1,TSUnmapped00000817.hg.1,TSUnmapped00000818.hg.1,TSUnmapped00000819.hg.1,TSUnmapped00000820.hg.1,TSUnmapped00000821.hg.1,TSUnmapped00000822.hg.1,TSUnmapped00000823.hg.1
0,9SR35159A_G04.CEL.pimg,9SR35159A_G04.CEL.pimg,9SR35159A_G04.CEL.pimg,9SR35159A_G04.CEL,Clariom_S_Human_HT,07/03/2020,UASG-1467,Ischemic Stroke,Large vessel disease,Male,...,2.49253,2.41132,3.33450,2.13259,2.57130,8.51848,2.02642,2.83986,2.45988,4.66323
1,9SR35157A_A09.CEL.pimg,9SR35157A_A09.CEL.pimg,9SR35157A_A09.CEL.pimg,9SR35157A_A09.CEL,Clariom_S_Human_HT,06/30/2020,UASG-0162,Ischemic Stroke,Small vessel disease / Lacunar,Male,...,2.52214,2.58208,2.90031,2.25478,2.70673,8.49474,1.66142,2.42892,2.67258,4.67467
2,9SR35157A_G11.CEL.pimg,9SR35157A_G11.CEL.pimg,9SR35157A_G11.CEL.pimg,9SR35157A_G11.CEL,Clariom_S_Human_HT,06/30/2020,UASG-0181,Ischemic Stroke,Small vessel disease / Lacunar,Male,...,2.53402,2.37989,3.35462,1.67938,2.64475,8.84778,1.78275,3.62172,3.27673,4.68155
3,9SR35157A_D05.CEL.pimg,9SR35157A_D05.CEL.pimg,9SR35157A_D05.CEL.pimg,9SR35157A_D05.CEL,Clariom_S_Human_HT,06/30/2020,UASG-1127,Ischemic Stroke,Other,Female,...,2.26048,2.47972,3.27244,2.05983,2.54761,8.18061,1.73277,3.27272,2.62022,4.24615
4,9SR35157A_F05.CEL.pimg,9SR35157A_F05.CEL.pimg,9SR35157A_F05.CEL.pimg,9SR35157A_F05.CEL,Clariom_S_Human_HT,06/30/2020,UASG-0007,Ischemic Stroke,Cardioembolic,Male,...,2.91858,2.34173,3.55027,2.35425,2.69594,8.58109,1.96196,3.54381,3.14377,3.66505


shape: (229, 21621)


In [10]:
print("Shape:", df_txt.shape)

Shape: (229, 21621)


In [11]:
print("First 12 column names:", list(df_txt.columns[:12]))

First 12 column names: ['Original', 'Summarized', 'Difference', 'Filename', 'Chip Type', 'Scan Date', 'UASG', 'Final Diagnosis', 'Diagnosis Subtype', 'Sex', 'Age At Onset', 'Hypertension']


In [12]:
print("First 175 column names:", list(df_txt.columns[:175]))

First 175 column names: ['Original', 'Summarized', 'Difference', 'Filename', 'Chip Type', 'Scan Date', 'UASG', 'Final Diagnosis', 'Diagnosis Subtype', 'Sex', 'Age At Onset', 'Hypertension', 'Diabetes', 'Hypercholesterolemia', 'Insulin', 'Oral diabetes meds', 'Insulin.1', 'GLUCOSE', 'GLUCOSE2', 'GLUCOSE FASTING', 'HEMOGLOBIN A1C', 'WHITE BLOOD CELL COUNT', 'HEMOGLOBIN', 'HEMATOCRIT', 'PLATELET COUNT', 'BLOOD UREA', 'CREATININE BLOOD', 'ESTIMATED GFR', 'GLUCOSE.1', 'INR', 'aPTT', 'NEUTROPHIL PERCENT', 'LYMPHOCYTE PERCENT', 'MONOCYTE PERCENT', 'WHITE BLOOD CELL COUNT2', 'HEMOGLOBIN2', 'HEMATOCRIT2', 'PLATELET COUNT2', 'BLOOD UREA2', 'CREATININE BLOOD2', 'ESTIMATED GFR2', 'GLUCOSE2.1', 'INR2', 'aPTT2', 'NEUTROPHIL PERCENT2', 'LYMPHOCYTE PERCENT2', 'MONOCYTE PERCENT2', 'CHOLESTEROL', 'HDL', 'LDL', 'TRIGLYCERIDE', 'Anticoagulation', 'Final Diagnosis.1', 'Ischemic stroke/ TIA cause', 'PmHx-Stroke', 'Congestive heart failure', 'Chronic renal insufficiency', 'Cancer', 'Unusual Disese', 'tPA', '

In [13]:
print("Last 5 column names:", list(df_txt.columns[-5:]))

Last 5 column names: ['TSUnmapped00000819.hg.1', 'TSUnmapped00000820.hg.1', 'TSUnmapped00000821.hg.1', 'TSUnmapped00000822.hg.1', 'TSUnmapped00000823.hg.1']


In [15]:
display(df_txt.iloc[:, :12].head(3))

,Original,Summarized,Difference,Filename,Chip Type,Scan Date,UASG,Final Diagnosis,Diagnosis Subtype,Sex,Age At Onset,Hypertension
0,9SR35159A_G04.CEL.pimg,9SR35159A_G04.CEL.pimg,9SR35159A_G04.CEL.pimg,9SR35159A_G04.CEL,Clariom_S_Human_HT,07/03/2020,UASG-1467,Ischemic Stroke,Large vessel disease,Male,71,Yes
1,9SR35157A_A09.CEL.pimg,9SR35157A_A09.CEL.pimg,9SR35157A_A09.CEL.pimg,9SR35157A_A09.CEL,Clariom_S_Human_HT,06/30/2020,UASG-0162,Ischemic Stroke,Small vessel disease / Lacunar,Male,35,No
2,9SR35157A_G11.CEL.pimg,9SR35157A_G11.CEL.pimg,9SR35157A_G11.CEL.pimg,9SR35157A_G11.CEL,Clariom_S_Human_HT,06/30/2020,UASG-0181,Ischemic Stroke,Small vessel disease / Lacunar,Male,55,Yes


------------------------

In [19]:
# Peek at raw text to guess the delimiter + header row
with open(tga_csv, "rb") as f:         # open the file whose path is in tga_csv, read as bytes (not regular text)
    for i in range(5):                 # look at first 5 lines
        print(f.readline().decode("utf-8", "replace").rstrip("\n")) # turn those bytes into text using UTF-8

        # If the file has any weird characters that don’t fit UTF-8, "replace" puts a special � symbol instead of crashing.
        # .rstrip("\n") --“Remove the newline character at the end,” so the printed lines don’t have an extra blank line between them.

NameError: name 'tga_csv' is not defined

In [20]:
import os

tga_csv = FILES["tga_csv"] # Pulls a path string from a dict called FILES under the key "tga_csv"
assert os.path.exists(tga_csv), f"Missing file: {tga_csv}" # Stops the program if the file doesn’t exist.
print("Path:", tga_csv)  #Prints the file path
print("Size (MB):", round(os.path.getsize(tga_csv)/1e6, 3)) #Prints the file size

Path: /home/sima/git_projects/illumina-genotyping-pipeline/input_data/expression_microarray/TGA-cohort-BackUp.csv
Size (MB): 39.303


In [23]:
import pandas as pd

df_csv = pd.read_csv(
    tga_csv,
    sep=None,               # auto-detect delimiter
    engine="python",        # needed for sep=None or regex seps
    encoding="utf-8",       # try utf-8 first; fallback to latin-1 if errors
    on_bad_lines="warn"     # show malformed lines instead of crashing
)
print(df_csv.shape)
df_csv.head(5)


(229, 21628)


,Original,Summarized,Difference,Filename,Chip Type,Scan Date,UASG,Final Diagnosis,Diagnosis Subtype,Sex,...,TSUnmapped00000810.hg.1,TSUnmapped00000812.hg.1,TSUnmapped00000815.hg.1,TSUnmapped00000817.hg.1,TSUnmapped00000818.hg.1,TSUnmapped00000819.hg.1,TSUnmapped00000820.hg.1,TSUnmapped00000821.hg.1,TSUnmapped00000822.hg.1,TSUnmapped00000823.hg.1
0,9SR35159A_G04.CEL.pimg,9SR35159A_G04.CEL.pimg,9SR35159A_G04.CEL.pimg,9SR35159A_G04.CEL,Clariom_S_Human_HT,07/03/2020,UASG-1467,Ischemic Stroke,Large vessel disease,Male,...,2.83986,2.45988,4.66323,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,9SR35157A_A09.CEL.pimg,9SR35157A_A09.CEL.pimg,9SR35157A_A09.CEL.pimg,9SR35157A_A09.CEL,Clariom_S_Human_HT,06/30/2020,UASG-0162,Ischemic Stroke,Small vessel disease / Lacunar,Male,...,2.42892,2.67258,4.67467,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,9SR35157A_G11.CEL.pimg,9SR35157A_G11.CEL.pimg,9SR35157A_G11.CEL.pimg,9SR35157A_G11.CEL,Clariom_S_Human_HT,06/30/2020,UASG-0181,Ischemic Stroke,Small vessel disease / Lacunar,Male,...,3.62172,3.27673,4.68155,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,9SR35157A_D05.CEL.pimg,9SR35157A_D05.CEL.pimg,9SR35157A_D05.CEL.pimg,9SR35157A_D05.CEL,Clariom_S_Human_HT,06/30/2020,UASG-1127,Ischemic Stroke,Other,Female,...,3.27272,2.62022,4.24615,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,9SR35157A_F05.CEL.pimg,9SR35157A_F05.CEL.pimg,9SR35157A_F05.CEL.pimg,9SR35157A_F05.CEL,Clariom_S_Human_HT,06/30/2020,UASG-0007,Ischemic Stroke,Cardioembolic,Male,...,3.54381,3.14377,3.66505,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
# how many unique IDs ins UASG column
print("UASG" in df_csv.columns)  
print(df_csv["UASG"].head())

# Count total vs unique
total_ids = df_csv["UASG"].shape[0]
unique_ids = df_csv["UASG"].nunique()
print(f"total_ids: {total_ids}")
print(f"unique_ids: {unique_ids}")

True
0    UASG-1467
1    UASG-0162
2    UASG-0181
3    UASG-1127
4    UASG-0007
Name: UASG, dtype: object
total_ids: 229
unique_ids: 229


In [25]:
print(df_csv["Final Diagnosis"].head())

0    Ischemic Stroke
1    Ischemic Stroke
2    Ischemic Stroke
3    Ischemic Stroke
4    Ischemic Stroke
Name: Final Diagnosis, dtype: object


In [26]:
# Count unique categories
n_categories = df_csv["Final Diagnosis"].nunique()
print("Number of unique Final Diagnosis categories:", n_categories)

Number of unique Final Diagnosis categories: 5


In [27]:
# List them all with counts
print(df_csv["Final Diagnosis"].value_counts())

Final Diagnosis
Ischemic Stroke       170
Control                54
TIA                     3
Hemorrhagic Stroke      1
TIA vs Control          1
Name: count, dtype: int64


In [62]:
# If you only want the unique category names:
print(df_csv["Final Diagnosis"].unique())

['Ischemic Stroke' 'Control' 'Hemorrhagic Stroke' 'TIA' 'TIA vs Control']


-----------

In [28]:
import os

transpose_csv = FILES["transpose_csv"] # Pulls a path string from a dict called FILES under the key "tga_csv"
assert os.path.exists(transpose_csv), f"Missing file: {transpose_csv}" # Stops the program if the file doesn’t exist.
print("Path:", transpose_csv)  #Prints the file path
print("Size (MB):", round(os.path.getsize(transpose_csv)/1e6, 3)) #Prints the file size

Path: /home/sima/git_projects/illumina-genotyping-pipeline/input_data/expression_microarray/transpose_numbers.csv
Size (MB): 46.415


In [29]:
import pandas as pd

df_transpose = pd.read_csv(
    transpose_csv,
    sep=None,               # auto-detect delimiter
    engine="python",        # needed for sep=None or regex seps
    encoding="utf-8",       # try utf-8 first; fallback to latin-1 if errors
    on_bad_lines="warn"     # show malformed lines instead of crashing
)
print(df_transpose.shape)
df_transpose.head(5)


(21449, 234)


,ID,Gene Symbol,unigene,gene_assignment,RefSeq,swissprot,unigene.1,UASG-0432,UASG-1331,UASG-1226,...,UASG-0202,UASG-0221,UASG-0304,UASG-0342,UASG-0361,UASG-1027,UASG-1155,UASG-1325,UASG-1420,UASG-0220
0,mRS good vs bad,NaN,NaN,NaN,NaN,NaN,NaN,1.00000,1.00000,1.00000,...,1.00000,1.00000,1.00000,1.00000,1.00000,1.00000,1.00000,1.00000,1.00000,1.00000
1,TC1000010642.hg.1,A1CF,NM_001198818 // Hs.282795 // connective tissue...,NM_001198818 // A1CF // APOBEC1 complementatio...,NM_001198818,NM_001198818 // Q9NQ94 /// NM_001198818 // Q7Z...,NM_001198818 // Hs.282795 // connective tissue...,1.62489,1.94362,1.84973,...,1.46509,1.55474,1.51391,1.53876,1.72148,1.61030,1.88767,1.84530,1.61760,1.64039
2,TC1600006789.hg.1,A2BP1,---,A2BP1.qAug10-unspliced // A2BP1 // Transcript ...,A2BP1.qAug10-unspliced,---,---,1.73784,1.31067,1.53732,...,1.12328,1.43262,1.23415,1.11851,1.23606,1.63694,1.27302,1.51636,1.27112,1.33102
3,TC1600006790.hg.1,A2BP1,---,A2BP1.tAug10-unspliced // A2BP1 // Transcript ...,A2BP1.tAug10-unspliced,---,---,1.93455,1.80662,1.91982,...,1.83373,1.67877,2.28307,1.77295,1.68748,1.77301,1.83773,1.87122,2.70517,1.57111
4,TC1600006812.hg.1,A2BP1,---,A2BP1.sAug10-unspliced // A2BP1 // Transcript ...,A2BP1.sAug10-unspliced,---,---,2.78974,2.72053,3.19491,...,2.84769,3.14492,3.28217,2.97546,2.91276,3.02366,3.06257,2.92001,2.98420,2.49120


In [47]:
print("First 175 column names:", list(df_transpose.columns[:234]))

First 175 column names: ['ID', 'Gene Symbol', 'unigene', 'gene_assignment', 'RefSeq', 'swissprot', 'unigene.1', 'UASG-0432', 'UASG-1331', 'UASG-1226', 'UASG-1231', 'UASG-0119', 'UASG-0025', 'UASG-1185', 'UASG-1163', 'UASG-1230', 'UASG-1366', 'UASG-1563', 'UASG-1141', 'UASG-1126', 'UASG-1504', 'UASG-1318', 'UASG-1418', 'UASG-0115', 'UASG-1168', 'UASG-1110', 'UASG-1035', 'UASG-1381', 'UASG-1184', 'UASG-1374', 'UASG-1127', 'UASG-0038', 'UASG-0313', 'UASG-0154', 'UASG-0006', 'UASG-1109', 'UASG-1312', 'UASG-1412', 'UASG-1067', 'UASG-0049', 'UASG-1042', 'UASG-1291', 'UASG-0063', 'UASG-0271', 'UASG-0010', 'UASG-1061', 'UASG-0052', 'UASG-0092', 'UASG-0316', 'UASG-1187', 'UASG-0197', 'UASG-1450', 'UASG-1094', 'UASG-1056', 'UASG-0239', 'UASG-0153', 'UASG-1260', 'UASG-1259', 'UASG-1032', 'UASG-1030', 'UASG-1143', 'UASG-0134', 'UASG-0157', 'UASG-0234', 'UASG-1466', 'UASG-0144', 'UASG-1054', 'UASG-1069', 'UASG-1360', 'UASG-1404', 'UASG-0004', 'UASG-1052', 'UASG-1467', 'UASG-0162', 'UASG-0170', 'UAS

In [48]:
print("UASG columns", list(df_transpose.columns[7:234]))

First 175 column names: ['UASG-0432', 'UASG-1331', 'UASG-1226', 'UASG-1231', 'UASG-0119', 'UASG-0025', 'UASG-1185', 'UASG-1163', 'UASG-1230', 'UASG-1366', 'UASG-1563', 'UASG-1141', 'UASG-1126', 'UASG-1504', 'UASG-1318', 'UASG-1418', 'UASG-0115', 'UASG-1168', 'UASG-1110', 'UASG-1035', 'UASG-1381', 'UASG-1184', 'UASG-1374', 'UASG-1127', 'UASG-0038', 'UASG-0313', 'UASG-0154', 'UASG-0006', 'UASG-1109', 'UASG-1312', 'UASG-1412', 'UASG-1067', 'UASG-0049', 'UASG-1042', 'UASG-1291', 'UASG-0063', 'UASG-0271', 'UASG-0010', 'UASG-1061', 'UASG-0052', 'UASG-0092', 'UASG-0316', 'UASG-1187', 'UASG-0197', 'UASG-1450', 'UASG-1094', 'UASG-1056', 'UASG-0239', 'UASG-0153', 'UASG-1260', 'UASG-1259', 'UASG-1032', 'UASG-1030', 'UASG-1143', 'UASG-0134', 'UASG-0157', 'UASG-0234', 'UASG-1466', 'UASG-0144', 'UASG-1054', 'UASG-1069', 'UASG-1360', 'UASG-1404', 'UASG-0004', 'UASG-1052', 'UASG-1467', 'UASG-0162', 'UASG-0170', 'UASG-0262', 'UASG-0173', 'UASG-0189', 'UASG-1040', 'UASG-0257', 'UASG-0344', 'UASG-0005', 

---------------------------

In [30]:
import os

snp_csv = FILES["snp_csv"]                                 # Pulls a path string from a dict called FILES under the key "tga_csv"
assert os.path.exists(snp_csv), f"Missing file: {snp_csv}" # Stops the program if the file doesn’t exist.
print("Path:", snp_csv)  #Prints the file path
print("Size (MB):", round(os.path.getsize(snp_csv)/1e6, 3)) #Prints the file size

Path: /home/sima/git_projects/illumina-genotyping-pipeline/input_data/sample_sheet/All_samples_Examine_SNPs_GWAS studies_GJ-P01_infiniumSampleSheet.csv
Size (MB): 0.029


In [32]:
import pandas as pd

# --- read df_snp ---
df_snp = pd.read_csv(
    snp_csv,
    sep=None,             # auto-detect delimiter
    engine="python",      # required for sep=None
    encoding="utf-8",
    on_bad_lines="warn",
    skiprows=8            # make the 9th line the header row
)


In [33]:
# Normalize column names (trim stray spaces, etc.)
df_snp.columns = df_snp.columns.str.strip()

In [34]:
# Sanity check: make sure Sample_Name exists
if "Sample_Name" not in df_snp.columns:
    print("Columns seen:", list(df_snp.columns)[:20])
    raise KeyError("Expected 'Sample_Name' in header after skiprows=8")

print("df_snp shape:", df_snp.shape)
display(df_snp.head(5))

df_snp shape: (288, 11)


,Sample_Name,Sample_ID,Sample_Plate,Sample_Well,SentrixBarcode_A,SentrixPosition_A,Gender,Sample_Group,Replicate,Parent1,Parent2
0,NA10837,22684#NA10837#207363850074#R12C02,WG0203968-MSA3,H12,207363850074,R12C02,Male,50.00,NaN,NaN,NaN
1,UASG-0400,22684#UASG-0400#207339170112#R01C01,WG0203968-MSA3,E01,207339170112,R01C01,Female,76.61,NaN,NaN,NaN
2,UASG-0002,22684#UASG-0002#207363850013#R01C01,WG0203968-MSA3,A01,207363850013,R01C01,Female,68.18,NaN,NaN,NaN
3,UASG-0399,22684#UASG-0399#207363850013#R02C01,WG0203968-MSA3,A02,207363850013,R02C01,Male,61.75,NaN,NaN,NaN
4,UASG-0013,22684#UASG-0013#207363850013#R03C01,WG0203968-MSA3,A03,207363850013,R03C01,Female,70.06,NaN,NaN,NaN


In [36]:
snp_ids = (df_snp["Sample_Name"]
           .astype(str)                   # make every value a string (even NaN -> "nan")
           .str.strip()                   # trim leading/trailing spaces
           .replace({"": pd.NA,           # turn empty strings into NA
                     "nan": pd.NA,        # because astype(str) made NaN into "nan"
                     "None": pd.NA})      # common literal text for missing
           .dropna()                      # drop the missing values
           .unique())                     # keep one of each remaining ID


In [37]:
print("Unique UASG in df_snp:", len(snp_ids))

Unique UASG in df_snp: 288


In [38]:
# Count how many start with “UASG”
col = (df_snp["Sample_Name"]
       .astype(str)
       .str.strip())

mask_uasg = col.str.startswith("UASG", na=False)           # case-sensitive
# or case-insensitive:
# mask_uasg = col.str.match(r'(?i)^UASG', na=False)

n_total = col.replace({"": pd.NA, "nan": pd.NA, "None": pd.NA}).dropna().shape[0]
n_uasg  = mask_uasg.sum()
n_other = n_total - n_uasg

print("Total non-missing Sample_Name:", n_total)
print("Start with UASG:", n_uasg)
print("Other IDs:", n_other)


Total non-missing Sample_Name: 288
Start with UASG: 285
Other IDs: 3


In [39]:
other_examples = (col[~mask_uasg]
                  .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
                  .dropna()
                  .unique()[:25])
print("Examples of non-UASG IDs:", other_examples)


Examples of non-UASG IDs: ['NA10837' 'NA12239' 'NA12146']


In [41]:
# --- extract clean UASG IDs from the main df (which has 'UASG' column) ---
cohort_ids = (df_csv["UASG"]
              .astype(str)
              .str.strip()
              .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
              .dropna()
              .unique())

print("Unique UASG in cohort df:", len(cohort_ids))

Unique UASG in cohort df: 229


In [42]:
# --- compare sets ---
snp_only = sorted(set(snp_ids) - set(cohort_ids))
cohort_only = sorted(set(cohort_ids) - set(snp_ids))
both = len(set(snp_ids) & set(cohort_ids))

print("In both:", both)
print("Only in df_snp:", len(snp_only))
print("Only in cohort df:", len(cohort_only))

In both: 226
Only in df_snp: 62
Only in cohort df: 3


In [91]:
# Peek at a few missing IDs
print("Example IDs only in df_snp:", snp_only[:10])
print("Example IDs only in cohort df:", cohort_only[:10])

Example IDs only in df_snp: ['NA10837', 'NA12146', 'NA12239', 'UASG-0002', 'UASG-0022', 'UASG-0030', 'UASG-0040', 'UASG-0043', 'UASG-0050', 'UASG-0051']
Example IDs only in cohort df: ['UASG-0129', 'UASG-0191', 'UASG-1168']


In [43]:
# Optional: nice summary table
summary = pd.DataFrame({
    "Status": ["In both", "Only in df_snp", "Only in cohort df"],
    "Count":  [both, len(snp_only), len(cohort_only)]
})
display(summary)

,Status,Count
0,In both,226
1,Only in df_snp,62
2,Only in cohort df,3


In [94]:
print(df_csv["Final Diagnosis"].unique())

['Ischemic Stroke' 'Control' 'Hemorrhagic Stroke' 'TIA' 'TIA vs Control']


In [96]:
print(df_csv["Final Diagnosis"].value_counts())

Final Diagnosis
Ischemic Stroke       170
Control                54
TIA                     3
Hemorrhagic Stroke      1
TIA vs Control          1
Name: count, dtype: int64


In [97]:
n_controls = (df_csv["Final Diagnosis"] == "Control").sum()
n_noncontrols = (df_csv["Final Diagnosis"] != "Control").sum()

print("Controls:", n_controls)
print("Non-controls:", n_noncontrols)


Controls: 54
Non-controls: 175


-------------------

### lET'S EXPLORE TRANSPOSE CSV (df_transpose) AND SNP DATASET (df_snp)

**df_transpose** has sample columns starting at column index 7 (8th column).  
**df_snp** is already loaded and has a Sample_Name column.

In [99]:
import pandas as pd

# --- helper: clean ID strings uniformly ---
def clean_ids(s):
    return (pd.Series(s, dtype="string")
              .str.strip()
              .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
              .dropna())

# 1) Get sample column names from df_transpose (from 8th col onward)
sample_cols = pd.Index(df_transpose.columns[7:])

# Clean them
sample_ids = clean_ids(sample_cols)

# 2) Count how many start with UASG (case-insensitive optional)
mask_uasg = sample_ids.str.match(r'(?i)^UASG')   # case-insensitive
n_total   = sample_ids.size
n_uasg    = mask_uasg.sum()
n_other   = n_total - n_uasg

print("df_transpose sample columns (from col 8):", n_total)
print("Start with UASG:", n_uasg)
print("Other IDs:", n_other)

# Peek examples
print("Examples UASG:", sample_ids[mask_uasg].unique()[:10].tolist())
print("Examples non-UASG:", sample_ids[~mask_uasg].unique()[:10].tolist())

# 3) Overlap with df_snp Sample_Name
# (recompute clean set if not already computed)
if "Sample_Name" not in df_snp.columns:
    raise KeyError("df_snp is missing 'Sample_Name'")

snp_ids = clean_ids(df_snp["Sample_Name"]).unique()

# Use sets for overlap math
set_transpose = set(sample_ids.unique())
set_snp       = set(snp_ids)

overlap        = set_transpose & set_snp
only_transpose = set_transpose - set_snp
only_snp       = set_snp - set_transpose

print("Overlap count (df_transpose ∩ df_snp):", len(overlap))
print("Only in df_transpose:", len(only_transpose))
print("Only in df_snp:", len(only_snp))

# 4) Optional: quick summary table
summary = pd.DataFrame({
    "Metric": [
        "df_transpose samples (from col 8)",
        "Start with UASG",
        "Other IDs",
        "Overlap with df_snp",
        "Only in df_transpose",
        "Only in df_snp",
    ],
    "Count": [n_total, n_uasg, n_other, len(overlap), len(only_transpose), len(only_snp)],
})
display(summary)


df_transpose sample columns (from col 8): 227
Start with UASG: 227
Other IDs: 0
Examples UASG: ['UASG-0432', 'UASG-1331', 'UASG-1226', 'UASG-1231', 'UASG-0119', 'UASG-0025', 'UASG-1185', 'UASG-1163', 'UASG-1230', 'UASG-1366']
Examples non-UASG: []
Overlap count (df_transpose ∩ df_snp): 224
Only in df_transpose: 3
Only in df_snp: 64


,Metric,Count
0,df_transpose samples (from col 8),227
1,Start with UASG,227
2,Other IDs,0
3,Overlap with df_snp,224
4,Only in df_transpose,3
5,Only in df_snp,64


In [100]:
print(only_transpose)

{'UASG-0129', 'UASG-0191', 'UASG-1168'}


In [101]:
print(df_transpose.columns[:7].tolist())


['ID', 'Gene Symbol', 'unigene', 'gene_assignment', 'RefSeq', 'swissprot', 'unigene.1']


------------------

### LET'S COMPARE df_csv and df_transpose

#### Pre-processing

In [109]:
# Find first column that matches the UASG pattern
uasg_start_idx = next(
    (i for i, c in enumerate(df_transpose.columns) if str(c).startswith("UASG")),
    None
)
print("First UASG column index:", uasg_start_idx)


First UASG column index: 7


In [127]:
# Step 1. Find where expression starts
# Find the first column that looks like a probe ID
expr_start_idx = df_csv.columns.get_loc("TC0100006437.hg.1")
print("Expression starts at column index:", expr_start_idx)

# That means metadata columns are from 0 up to expr_start_idx-1
print("Number of metadata columns in df_csv:", expr_start_idx)
print("Example metadata columns:", df_csv.columns[:10].tolist())



Expression starts at column index: 180
Number of metadata columns in df_csv: 180
Example metadata columns: ['Original', 'Summarized', 'Difference', 'Filename', 'Chip Type', 'Scan Date', 'UASG', 'Final Diagnosis', 'Diagnosis Subtype', 'Sex']


#### Step 1. Get sample IDs from each dataset

In [110]:
# Clean sample IDs in df_csv (from the UASG column)
csv_ids = (df_csv["UASG"]
           .astype(str).str.strip()
           .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
           .dropna()
           .unique())

# Clean sample IDs in df_transpose (from column headers after col 7)
transpose_ids = (pd.Index(df_transpose.columns[7:])
                 .astype(str)
                 .str.strip()
                 .dropna()
                 .unique())


#### Step 2. Compare overlaps and missing

In [111]:
set_csv       = set(csv_ids)
set_transpose = set(transpose_ids)

overlap        = set_csv & set_transpose
only_csv       = set_csv - set_transpose
only_transpose = set_transpose - set_csv

print("Total in df_csv:", len(set_csv))
print("Total in df_transpose:", len(set_transpose))
print("Overlap:", len(overlap))
print("Only in df_csv (missing in transpose):", len(only_csv))
print("Only in df_transpose (unexpected):", len(only_transpose))

# See the actual missing IDs
print("IDs only in df_csv:", only_csv)
print("IDs only in df_transpose:", only_transpose)


Total in df_csv: 229
Total in df_transpose: 227
Overlap: 227
Only in df_csv (missing in transpose): 2
Only in df_transpose (unexpected): 0
IDs only in df_csv: {'UASG-1436', 'UASG-0256'}
IDs only in df_transpose: set()


----------------

## UASG-1436

In [ ]:
target = "UASG-1436"

# Normalize UASG column a bit to avoid whitespace issues
df_csv["UASG_clean"] = df_csv["UASG"].astype(str).str.strip()

# Rows for this sample
row_mask = df_csv["UASG_clean"] == target
print("Rows matching exact 'UASG-1436':", row_mask.sum())

sample_row = df_csv.loc[row_mask]
display(sample_row.iloc[:,:30])  # peek first ~30 metadata columns


In [118]:
META_N = 175
expr_cols = df_csv.columns[META_N:]

if not sample_row.empty:
    expr_vals = sample_row[expr_cols]
    n_total = expr_vals.shape[1]
    n_na = expr_vals.isna().sum(axis=1).iloc[0]
    n_zero = (expr_vals.fillna(0) == 0).sum(axis=1).iloc[0]
    print(f"Expr columns: {n_total} | NAs: {n_na} | Zeros: {n_zero}")

    # Basic variability check (all-constant rows often get filtered upstream)
    n_non_const = (expr_vals.nunique(axis=1) > 1).sum()
    print("Row shows more than one unique expression value?", bool(n_non_const))

Expr columns: 21454 | NAs: 7 | Zeros: 7
Row shows more than one unique expression value? True


In [119]:
if "Sample_Name" in df_snp.columns:
    snp_ids = df_snp["Sample_Name"].astype(str).str.strip().unique()
    in_snp = target in set(snp_ids)
    print("UASG-1436 in df_snp?", in_snp)


UASG-1436 in df_snp? True


In [120]:
if not sample_row.empty:
    na_rate = n_na / max(n_total, 1)
    print(f"Approx NA rate for UASG-1436: {na_rate:.3%}")


Approx NA rate for UASG-1436: 0.033%


## UASG-0256

In [121]:
target = "UASG-0256"

# Normalize UASG column a bit to avoid whitespace issues
df_csv["UASG_clean"] = df_csv["UASG"].astype(str).str.strip()

# Rows for this sample
row_mask = df_csv["UASG_clean"] == target
print("Rows matching exact 'UASG-1436':", row_mask.sum())

sample_row = df_csv.loc[row_mask]
display(sample_row.iloc[:,:30])  # peek first ~30 metadata columns


Rows matching exact 'UASG-1436': 1


,Original,Summarized,Difference,Filename,Chip Type,Scan Date,UASG,Final Diagnosis,Diagnosis Subtype,Sex,...,HEMOGLOBIN A1C,WHITE BLOOD CELL COUNT,HEMOGLOBIN,HEMATOCRIT,PLATELET COUNT,BLOOD UREA,CREATININE BLOOD,ESTIMATED GFR,GLUCOSE.1,INR
228,9SR35158A_H03.CEL.pimg,9SR35158A_H03.CEL.pimg,9SR35158A_H03.CEL.pimg,9SR35158A_H03.CEL,Clariom_S_Human_HT,07/01/2020,UASG-0256,TIA vs Control,NaN,Female,...,?,6,131,0.4,228,?,101,?,7,1.4


In [122]:
META_N = 175
expr_cols = df_csv.columns[META_N:]

if not sample_row.empty:
    expr_vals = sample_row[expr_cols]
    n_total = expr_vals.shape[1]
    n_na = expr_vals.isna().sum(axis=1).iloc[0]
    n_zero = (expr_vals.fillna(0) == 0).sum(axis=1).iloc[0]
    print(f"Expr columns: {n_total} | NAs: {n_na} | Zeros: {n_zero}")

    # Basic variability check (all-constant rows often get filtered upstream)
    n_non_const = (expr_vals.nunique(axis=1) > 1).sum()
    print("Row shows more than one unique expression value?", bool(n_non_const))

Expr columns: 21454 | NAs: 7 | Zeros: 7
Row shows more than one unique expression value? True


In [123]:
if "Sample_Name" in df_snp.columns:
    snp_ids = df_snp["Sample_Name"].astype(str).str.strip().unique()
    in_snp = target in set(snp_ids)
    print("UASG-1436 in df_snp?", in_snp)


UASG-1436 in df_snp? True


In [124]:
if not sample_row.empty:
    na_rate = n_na / max(n_total, 1)
    print(f"Approx NA rate for UASG-1436: {na_rate:.3%}")


Approx NA rate for UASG-1436: 0.033%


---------------------

## Check and Merge Metadata

#### Step 1. Collect metadata column names

In [130]:
meta_csv = list(df_csv.columns[:180])
meta_transpose = list(df_transpose.columns[:7])

print("df_csv metadata cols:", len(meta_csv))
print("df_transpose metadata cols:", len(meta_transpose))
print("First few from df_csv:", meta_csv[:15])
print("All from df_transpose:", meta_transpose)

df_csv metadata cols: 180
df_transpose metadata cols: 7
First few from df_csv: ['Original', 'Summarized', 'Difference', 'Filename', 'Chip Type', 'Scan Date', 'UASG', 'Final Diagnosis', 'Diagnosis Subtype', 'Sex', 'Age At Onset', 'Hypertension', 'Diabetes', 'Hypercholesterolemia', 'Insulin']
All from df_transpose: ['ID', 'Gene Symbol', 'unigene', 'gene_assignment', 'RefSeq', 'swissprot', 'unigene.1']


#### Step 2. Compare overlap

In [131]:
common_meta = set(meta_csv) & set(meta_transpose)
only_in_csv = set(meta_csv) - set(meta_transpose)
only_in_transpose = set(meta_transpose) - set(meta_csv)

print("Common metadata:", common_meta)
print("Unique to df_csv:", list(only_in_csv)[:20])   # peek at 20
print("Unique to df_transpose:", list(only_in_transpose))

Common metadata: set()
Unique to df_csv: ['Scan Date', 'Hemorrhagic Conversion', 'Type of Event #2', 'GLUCOSE FASTING', 'Other PmHx 1', 'Coumadin', 'Sex', 'Pt accompanied with #3:', 'PmX-TIA', 'GLUCOSE.1', 'LVH', 'DOAC', 'Final Diagnosis', 'Diabetes', 'New Problems or Diagnoses Since Enrolled #2', 'Date of Death.1', ' and Location of Event #2', 'Notes or Comments #1', 'Home Time.1', 'Cancer']
Unique to df_transpose: ['Gene Symbol', 'unigene', 'RefSeq', 'swissprot', 'ID', 'gene_assignment', 'unigene.1']


#### 1. Keep the first row as sample-level clinical annotation

In [132]:
# First row (row 0) is clinical info across the sample columns
clinical_row = df_transpose.iloc[0]

print("Clinical row preview:")
print(clinical_row.iloc[7:15])  # show for first few samples

# Store it as a separate metadata dataframe
clinical_meta = clinical_row[7:].to_frame(name="mRS_status")
clinical_meta.index.name = "SampleID"

print("clinical_meta shape:", clinical_meta.shape)
display(clinical_meta.head())

Clinical row preview:
UASG-0432    1.0
UASG-1331    1.0
UASG-1226    1.0
UASG-1231    1.0
UASG-0119      ?
UASG-0025    1.0
UASG-1185    1.0
UASG-1163    1.0
Name: 0, dtype: object
clinical_meta shape: (227, 1)


,mRS_status
SampleID,
UASG-0432,1.0
UASG-1331,1.0
UASG-1226,1.0
UASG-1231,1.0
UASG-0119,?


#### 2. Expression matrix starts at row 1 onward

In [133]:
# Expression data starts at row 1 (probes in 'ID', samples in cols 7+)
expr_T = df_transpose.iloc[1:].set_index("ID")

print("Expression shape:", expr_T.shape)  # rows = probes, cols = samples

Expression shape: (21448, 233)


-------

In [144]:
df_csv = df_csv.drop(columns=["UASG_clean"], errors="ignore")

In [145]:
# Probe IDs from df_csv
# Expression starts at index 180, so all columns from 180 onward are probe IDs:
probe_ids_csv = df_csv.columns[180:].astype(str).str.strip()
print("Number of probes in df_csv:", len(probe_ids_csv))

Number of probes in df_csv: 21448


In [146]:
# Probe IDs from df_transpose
# They live in the "ID" column, starting from row 1 onward (row 0 is your clinical annotation):
probe_ids_transpose = (
    df_transpose.loc[1:, "ID"]   # skip row 0
    .astype(str)
    .str.strip()
    .unique()
)
print("Number of probes in df_transpose:", len(probe_ids_transpose))

Number of probes in df_transpose: 21448


#### Overlap between the two

In [147]:
set_csv = set(probe_ids_csv)
set_transpose = set(probe_ids_transpose)

overlap = set_csv & set_transpose
only_in_csv = set_csv - set_transpose
only_in_transpose = set_transpose - set_csv

print("Overlap probes:", len(overlap))
print("Only in df_csv:", len(only_in_csv))
print("Only in df_transpose:", len(only_in_transpose))

# peek at a few missing ones
print("Examples only in df_csv:", list(sorted(only_in_csv))[:10])
print("Examples only in df_transpose:", list(sorted(only_in_transpose))[:10])

Overlap probes: 21448
Only in df_csv: 0
Only in df_transpose: 0
Examples only in df_csv: []
Examples only in df_transpose: []


-------------------

we’ll make a copy of df_transpose and append the two missing sample columns (UASG-1436, UASG-0256) pulled from df_csv, aligning by probe IDs.

Assumptions (based on your notes):

In df_csv, expression starts at column index 180; columns 180: are probe IDs (e.g., TC...).

In df_transpose, column "ID" holds probe IDs; row 0 is a clinical annotation row you want to keep; rows 1+: expression; sample columns start at index 7.

We’ll keep the clinical row and set its values for the new samples to NA (you can fill later).

In [148]:
import pandas as pd

# --- parameters you already established ---
EXPR_START_CSV = 180
MISSING_SAMPLES = ["UASG-1436", "UASG-0256"]  # or compute dynamically from your prior sets

# --- make a safe copy ---
dfT2 = df_transpose.copy(deep=True)

# --- probe order from df_transpose (rows 1+ are expression) ---
probe_order = (dfT2.loc[1:, "ID"]
               .astype(str).str.strip().tolist())

# --- expression columns in df_csv (drop helper columns like 'UASG_clean' if present) ---
csv_expr_cols = (df_csv
                 .drop(columns=["UASG_clean"], errors="ignore")
                 .columns[EXPR_START_CSV:])

# Optional: restrict to probes that exist in df_transpose to ensure perfect alignment
csv_expr_cols = [c for c in csv_expr_cols if c in set(probe_order)]

# --- make a fast index on UASG for df_csv ---
df_csv["_UASG_idx_"] = df_csv["UASG"].astype(str).str.strip()
csv_by_uasg = df_csv.set_index("_UASG_idx_", drop=False)

added = []
skipped = []

for sid in MISSING_SAMPLES:
    if sid not in csv_by_uasg.index:
        skipped.append((sid, "not found in df_csv['UASG']"))
        continue

    # 1) get that sample's expression row from df_csv and align to df_transpose probe order
    row = csv_by_uasg.loc[sid, csv_expr_cols]          # series: probes → values
    expr_vec = row.reindex(probe_order)                 # align exactly to df_transpose's ID order

    # 2) add a new column to dfT2
    #    - clinical row (row 0): NA placeholder (fill later if you have the phenotype elsewhere)
    #    - expression rows (1:): the aligned vector
    dfT2[sid] = pd.NA
    dfT2.loc[1:, sid] = expr_vec.values

    added.append(sid)

# --- report ---
print(f"Added {len(added)} sample(s): {added}")
if skipped:
    print("Skipped:", skipped)

print("New df_transpose shape:", dfT2.shape)

# (Optional) sanity checks
#  - confirm the new columns exist
print("New columns present?", all(sid in dfT2.columns for sid in MISSING_SAMPLES))
#  - confirm first few values line up with a known probe
if probe_order:
    probe0 = probe_order[0]
    print("Example (first probe) values for added samples:")
    for sid in added:
        print(probe0, sid, "→", dfT2.loc[dfT2["ID"] == probe0, sid].iloc[0])


Added 2 sample(s): ['UASG-1436', 'UASG-0256']
New df_transpose shape: (21449, 236)
New columns present? True
Example (first probe) values for added samples:
TC1000010642.hg.1 UASG-1436 → 2.02925
TC1000010642.hg.1 UASG-0256 → 2.53735


In [44]:
# If you later want the expression-only matrix, use:
expr_T = dfT2.set_index("ID").iloc[1:]   # rows=probes, cols=samples


NameError: name 'dfT2' is not defined

In [150]:
import pandas as pd

# Pick a subset: first 6 rows (row 0 = clinical + first 5 probes)
# and first 10 sample columns plus the two new ones
cols_to_show = list(dfT2.columns[:12]) + ["UASG-1436", "UASG-0256"]

print("dfT2 preview:")
display(dfT2.loc[:5, ["ID"] + cols_to_show])


dfT2 preview:


,ID,ID,Gene Symbol,unigene,gene_assignment,RefSeq,swissprot,unigene.1,UASG-0432,UASG-1331,UASG-1226,UASG-1231,UASG-0119,UASG-1436,UASG-0256
0,mRS good vs bad,mRS good vs bad,NaN,NaN,NaN,NaN,NaN,NaN,1.00000,1.00000,1.00000,1.00000,?,<NA>,<NA>
1,TC1000010642.hg.1,TC1000010642.hg.1,A1CF,NM_001198818 // Hs.282795 // connective tissue...,NM_001198818 // A1CF // APOBEC1 complementatio...,NM_001198818,NM_001198818 // Q9NQ94 /// NM_001198818 // Q7Z...,NM_001198818 // Hs.282795 // connective tissue...,1.62489,1.94362,1.84973,1.87490,1.6582,2.02925,2.53735
2,TC1600006789.hg.1,TC1600006789.hg.1,A2BP1,---,A2BP1.qAug10-unspliced // A2BP1 // Transcript ...,A2BP1.qAug10-unspliced,---,---,1.73784,1.31067,1.53732,1.49705,1.66407,6.77244,6.83558
3,TC1600006790.hg.1,TC1600006790.hg.1,A2BP1,---,A2BP1.tAug10-unspliced // A2BP1 // Transcript ...,A2BP1.tAug10-unspliced,---,---,1.93455,1.80662,1.91982,2.00325,1.6792,8.14672,8.42271
4,TC1600006812.hg.1,TC1600006812.hg.1,A2BP1,---,A2BP1.sAug10-unspliced // A2BP1 // Transcript ...,A2BP1.sAug10-unspliced,---,---,2.78974,2.72053,3.19491,3.33918,2.89034,8.103,8.21649
5,TC1200009847.hg.1,TC1200009847.hg.1,A2M,NM_000014 // Hs.212838 // adipose tissue| adre...,NM_000014 // A2M // alpha-2-macroglobulin // 1...,NM_000014,NM_000014 // P01023 /// ENST00000318602 // P01...,NM_000014 // Hs.212838 // adipose tissue| adre...,3.86055,4.07226,4.42702,3.58783,3.74066,2.18717,1.44247


In [ ]:
# sample overlap between df_snp and dfT2

In [151]:
import pandas as pd

def clean_ids(iterable):
    s = pd.Series(iterable, dtype="string")
    return (s.str.strip()
             .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
             .dropna())

# 1) Sample IDs from df_snp
snp_ids = clean_ids(df_snp["Sample_Name"]).unique()
set_snp = set(snp_ids)

# 2) Sample IDs from dfT2 (columns 7 onwards are samples)
t2_sample_cols = dfT2.columns[7:]
t2_ids = clean_ids(t2_sample_cols).unique()
set_t2 = set(t2_ids)

# 3) Overlap math
overlap        = set_snp & set_t2
only_in_snp    = set_snp - set_t2
only_in_dfT2   = set_t2 - set_snp

print("Totals:")
print(" - df_snp samples:   ", len(set_snp))
print(" - dfT2 samples:     ", len(set_t2))
print(" - Overlap:          ", len(overlap))
print(" - Only in df_snp:   ", len(only_in_snp))
print(" - Only in dfT2:     ", len(only_in_dfT2))

# Peek some examples
print("\nExamples only in df_snp:", list(sorted(only_in_snp))[:15])
print("Examples only in dfT2:", list(sorted(only_in_dfT2))[:15])

# 4) Optional: how many start with UASG in each, and among overlaps
mask_uasg_snp  = pd.Series(list(set_snp), dtype="string").str.match(r'(?i)^UASG')
mask_uasg_t2   = pd.Series(list(set_t2),  dtype="string").str.match(r'(?i)^UASG')
mask_uasg_olap = pd.Series(list(overlap), dtype="string").str.match(r'(?i)^UASG')

print("\nUASG breakdown:")
print(" - df_snp UASG IDs:  ", int(mask_uasg_snp.sum()))
print(" - dfT2  UASG IDs:   ", int(mask_uasg_t2.sum()))
print(" - Overlap UASG IDs: ", int(mask_uasg_olap.sum()))

# 5) Nice summary table
summary = pd.DataFrame({
    "Metric": [
        "df_snp unique samples",
        "dfT2 unique samples",
        "Overlap",
        "Only in df_snp",
        "Only in dfT2",
    ],
    "Count": [len(set_snp), len(set_t2), len(overlap), len(only_in_snp), len(only_in_dfT2)]
})
display(summary)

# 6) Optional: verify the two previously-missing samples made it into dfT2
expected_new = {"UASG-1436", "UASG-0256"}
present_new  = expected_new & set_t2
print("\nPreviously-missing now present in dfT2:", present_new)
missing_still = expected_new - set_t2
if missing_still:
    print("Still missing from dfT2:", missing_still)


Totals:
 - df_snp samples:    288
 - dfT2 samples:      229
 - Overlap:           226
 - Only in df_snp:    62
 - Only in dfT2:      3

Examples only in df_snp: ['NA10837', 'NA12146', 'NA12239', 'UASG-0002', 'UASG-0022', 'UASG-0030', 'UASG-0040', 'UASG-0043', 'UASG-0050', 'UASG-0051', 'UASG-0071', 'UASG-0072', 'UASG-0074', 'UASG-0091', 'UASG-0094']
Examples only in dfT2: ['UASG-0129', 'UASG-0191', 'UASG-1168']

UASG breakdown:
 - df_snp UASG IDs:   285
 - dfT2  UASG IDs:    229
 - Overlap UASG IDs:  226


,Metric,Count
0,df_snp unique samples,288
1,dfT2 unique samples,229
2,Overlap,226
3,Only in df_snp,62
4,Only in dfT2,3



Previously-missing now present in dfT2: {'UASG-1436', 'UASG-0256'}


In [152]:
import pandas as pd

# --- helpers ---
def clean_ids(iterable):
    s = pd.Series(iterable, dtype="string")
    return (s.str.strip()
             .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
             .dropna())

# 1) Recompute the overlap set (robust to reruns)
snp_ids = set(clean_ids(df_snp["Sample_Name"]).unique())
t2_ids  = set(clean_ids(dfT2.columns[7:]).unique())
overlap_ids = snp_ids & t2_ids
print("Overlap (df_snp ∩ dfT2):", len(overlap_ids))

# 2) Build a metadata table from df_csv (UASG + diagnosis + sex)
meta_cols = ["UASG", "Final Diagnosis", "Sex"]
missing_cols = [c for c in meta_cols if c not in df_csv.columns]
if missing_cols:
    raise KeyError(f"df_csv missing expected columns: {missing_cols}")

meta = (df_csv[meta_cols]
        .drop_duplicates(subset=["UASG"])
        .assign(UASG=lambda d: d["UASG"].astype(str).str.strip())
        .set_index("UASG"))

# 3) Align metadata to *only* the overlapping sample IDs
#    (non-UASG IDs in the overlap will become NaN here, which is expected)
overlap_meta = meta.reindex(sorted(overlap_ids))

# 4) Derive a simple StrokeStatus (Control vs Non-control) from Final Diagnosis
#    Adjust the rule as needed (e.g., "Healthy Control", "HC" etc.)
fd = overlap_meta["Final Diagnosis"].astype("string")
is_control = fd.str.contains("control", case=False, na=False)
overlap_meta["StrokeStatus"] = pd.Series(pd.NA, index=overlap_meta.index)
overlap_meta.loc[is_control, "StrokeStatus"] = "Control"
overlap_meta.loc[~is_control & fd.notna(), "StrokeStatus"] = "Non-control"

# 5) Quick row preview
print("\nPreview (first 10):")
display(overlap_meta.head(10))

# 6) Counts by StrokeStatus and Sex
print("\nCounts by StrokeStatus:")
print(overlap_meta["StrokeStatus"].value_counts(dropna=False))

print("\nCounts by Sex:")
print(overlap_meta["Sex"].value_counts(dropna=False))

print("\nCross-tab (StrokeStatus × Sex):")
display(pd.crosstab(overlap_meta["StrokeStatus"], overlap_meta["Sex"], dropna=False))

# 7) Report IDs with missing metadata (e.g., non-UASG overlap or unmatched)
missing_meta = overlap_meta.index[overlap_meta["Final Diagnosis"].isna() | overlap_meta["Sex"].isna()]
print("\nIDs in overlap lacking diagnosis/sex in df_csv (first 20):")
print(list(missing_meta[:20]))


Overlap (df_snp ∩ dfT2): 226

Preview (first 10):


,Final Diagnosis,Sex,StrokeStatus
UASG,,,
UASG-0003,Ischemic Stroke,Male,Non-control
UASG-0004,TIA,Male,Non-control
UASG-0005,Ischemic Stroke,Female,Non-control
UASG-0006,Ischemic Stroke,Female,Non-control
UASG-0007,Ischemic Stroke,Male,Non-control
UASG-0008,Ischemic Stroke,Male,Non-control
UASG-0010,Ischemic Stroke,Male,Non-control
UASG-0013,Ischemic Stroke,Female,Non-control
UASG-0019,Ischemic Stroke,Male,Non-control



Counts by StrokeStatus:
StrokeStatus
Non-control    171
Control         55
Name: count, dtype: int64

Counts by Sex:
Sex
Male      126
Female    100
Name: count, dtype: int64

Cross-tab (StrokeStatus × Sex):


Sex,Female,Male
StrokeStatus,,
Control,31,24
Non-control,69,102



IDs in overlap lacking diagnosis/sex in df_csv (first 20):
[]


In [153]:
# overlap set from your earlier calculation
overlap_ids = set(df_snp["Sample_Name"].astype(str).str.strip()) & set(dfT2.columns[7:])
print("Overlap:", len(overlap_ids))


Overlap: 226


In [155]:
# pull metadata from df_csv
# Clean UASG in df_csv
df_csv["_UASG_clean"] = df_csv["UASG"].astype(str).str.strip()

# Subset to overlap samples
meta_overlap = df_csv.loc[df_csv["_UASG_clean"].isin(overlap_ids), 
                          ["UASG", "Final Diagnosis", "Sex"]]

print(meta_overlap.shape)
display(meta_overlap.head())


(226, 3)


,UASG,Final Diagnosis,Sex
0,UASG-1467,Ischemic Stroke,Male
1,UASG-0162,Ischemic Stroke,Male
2,UASG-0181,Ischemic Stroke,Male
3,UASG-1127,Ischemic Stroke,Female
4,UASG-0007,Ischemic Stroke,Male


In [156]:
# count categories
# Diagnosis counts
print("\nFinal Diagnosis counts:")
print(meta_overlap["Final Diagnosis"].value_counts())

# Sex counts
print("\nSex counts:")
print(meta_overlap["Sex"].value_counts())

# Cross-tab for combined view
print("\nCross-tab (Diagnosis × Sex):")
print(pd.crosstab(meta_overlap["Final Diagnosis"], meta_overlap["Sex"]))



Final Diagnosis counts:
Final Diagnosis
Ischemic Stroke       167
Control                54
TIA                     3
Hemorrhagic Stroke      1
TIA vs Control          1
Name: count, dtype: int64

Sex counts:
Sex
Male      126
Female    100
Name: count, dtype: int64

Cross-tab (Diagnosis × Sex):
Sex                 Female  Male
Final Diagnosis                 
Control                 30    24
Hemorrhagic Stroke       0     1
Ischemic Stroke         68    99
TIA                      1     2
TIA vs Control           1     0
